# HYDRUS1D Phase 3: Infiltration Dynamics Demo

This notebook demonstrates the Phase 3 Richards equation solver with:
- Modern xarray output format for easy data manipulation
- Spatiotemporal visualization (heatmaps showing dynamics over time and depth)
- Traditional profile and time series plots
- Mass balance analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Add packages to path
phase3_path = str(Path.cwd())
phase2_path = str(Path.cwd().parent / 'phase2')
sys.path.insert(0, phase2_path)
sys.path.insert(0, phase3_path)

# Import Phase 3
from hydrus1dpy import HydrusModel
from hydrus1dpy.materials import VanGenuchten

# Optional: xarray for modern data handling
try:
    import xarray as xr
    HAS_XARRAY = True
except ImportError:
    print("Warning: xarray not available. Install with: pip install xarray")
    HAS_XARRAY = False

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Setup Soil Properties and Model

We'll simulate infiltration into a loam soil using van Genuchten (1980) parameters.

In [ ]:
# Create van Genuchten hydraulic model for loam soil
# Parameters from Carsel & Parrish (1988)
vg_loam = VanGenuchten(
    theta_r=0.078,  # Residual water content
    theta_s=0.430,  # Saturated water content
    alpha=0.036,    # Scale parameter [1/cm]
    n=1.56,         # Shape parameter
    Ks=24.96,       # Saturated conductivity [cm/day]
    l=0.5           # Pore connectivity
)

print("Soil Hydraulic Properties (Loam):")
print(f"  θr = {vg_loam.theta_r:.3f}")
print(f"  θs = {vg_loam.theta_s:.3f}")
print(f"  α  = {vg_loam.alpha:.3f} 1/cm")
print(f"  n  = {vg_loam.n:.2f}")
print(f"  Ks = {vg_loam.Ks:.2f} cm/day")

In [ ]:
# Create model: 100 cm deep column with 101 nodes (1 cm spacing)
model = HydrusModel(
    depth=100.0,
    n_nodes=101,
    material=vg_loam
)

# Set boundary conditions
infiltration_rate = 5.0  # cm/day
model.set_top_bc('flux', flux=infiltration_rate)
model.set_bottom_bc('free_drainage')

# Set initial conditions: dry profile
model.set_initial_conditions('hydrostatic', h_bottom=-200)

print(f"\nModel Configuration:")
print(f"  Domain: {model.depth:.1f} cm, {model.n_nodes} nodes")
print(f"  Grid spacing: {model.depth/(model.n_nodes-1):.2f} cm")
print(f"  Top BC: {infiltration_rate} cm/day infiltration")
print(f"  Bottom BC: Free drainage")
print(f"  Initial condition: Hydrostatic (h_bottom = -200 cm)")

## 2. Run Simulation

We simulate 1 day of infiltration with adaptive time stepping.

In [ ]:
# Run simulation
results = model.run(
    t_end=1.0,
    dt_init=0.0005,
    dt_min=1e-6,
    dt_max=0.01,
    verbose=True
)

## 3. Convert to xarray for Easy Data Handling

xarray provides labeled, multi-dimensional arrays with powerful selection and plotting capabilities.

In [ ]:
if HAS_XARRAY:
    # Convert results to xarray Dataset
    ds = model.to_xarray()
    
    print("xarray Dataset:")
    print(ds)
    
    print("\nDataset attributes:")
    for key, value in ds.attrs.items():
        print(f"  {key}: {value}")
else:
    print("xarray not available - using dict results")
    ds = None

## 4. Spatiotemporal Visualization

Heatmaps show the evolution of state variables over time and depth - perfect for understanding infiltration dynamics!

In [ ]:
# Create spatiotemporal heatmaps
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if HAS_XARRAY and ds is not None:
    # Use xarray's built-in plotting (much easier!)
    
    # Water content heatmap
    ds.theta.plot(
        ax=axes[0],
        x='time',
        y='depth',
        cmap='Blues',
        cbar_kwargs={'label': 'Water content θ [-]'}
    )
    axes[0].set_title('Water Content Dynamics')
    axes[0].set_xlabel('Time [days]')
    axes[0].set_ylabel('Depth [cm]')
    axes[0].invert_yaxis()  # Surface at top
    
    # Pressure head heatmap
    ds.h.plot(
        ax=axes[1],
        x='time',
        y='depth',
        cmap='RdBu_r',
        vmin=-200,
        vmax=0,
        cbar_kwargs={'label': 'Pressure head h [cm]'}
    )
    axes[1].set_title('Pressure Head Dynamics')
    axes[1].set_xlabel('Time [days]')
    axes[1].set_ylabel('Depth [cm]')
    axes[1].invert_yaxis()
    
else:
    # Fallback to matplotlib if xarray not available
    times = results['times']
    depths = model.depths
    theta = results['theta'].T  # Transpose for imshow
    h = results['h'].T
    
    im1 = axes[0].imshow(
        theta,
        aspect='auto',
        extent=[times[0], times[-1], depths[-1], depths[0]],
        cmap='Blues',
        interpolation='bilinear'
    )
    plt.colorbar(im1, ax=axes[0], label='Water content θ [-]')
    axes[0].set_title('Water Content Dynamics')
    axes[0].set_xlabel('Time [days]')
    axes[0].set_ylabel('Depth [cm]')
    
    im2 = axes[1].imshow(
        h,
        aspect='auto',
        extent=[times[0], times[-1], depths[-1], depths[0]],
        cmap='RdBu_r',
        vmin=-200,
        vmax=0,
        interpolation='bilinear'
    )
    plt.colorbar(im2, ax=axes[1], label='Pressure head h [cm]')
    axes[1].set_title('Pressure Head Dynamics')
    axes[1].set_xlabel('Time [days]')
    axes[1].set_ylabel('Depth [cm]')

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Blue colors (left): Wetting front propagation downward")
print("- Red→Blue (right): Pressure head increase as water infiltrates")
print("- Sharp gradients show the wetting front location")

## 5. Profile Snapshots at Different Times

Traditional vertical profiles showing state at different time points.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Select time indices to plot
n_times = len(results['times'])
time_indices = [0, n_times//4, n_times//2, 3*n_times//4, -1]
colors = plt.cm.viridis(np.linspace(0, 1, len(time_indices)))

for idx, color in zip(time_indices, colors):
    t = results['times'][idx]
    h = results['h'][idx]
    theta = results['theta'][idx]
    
    axes[0].plot(theta, model.depths, color=color, label=f't = {t:.3f} d', linewidth=2)
    axes[1].plot(h, model.depths, color=color, label=f't = {t:.3f} d', linewidth=2)

# Water content profile
axes[0].axvline(vg_loam.theta_r, color='gray', linestyle='--', alpha=0.5, label='θr')
axes[0].axvline(vg_loam.theta_s, color='gray', linestyle='--', alpha=0.5, label='θs')
axes[0].set_xlabel('Water content θ [-]', fontsize=12)
axes[0].set_ylabel('Depth [cm]', fontsize=12)
axes[0].set_title('Water Content Profiles', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Pressure head profile
axes[1].set_xlabel('Pressure head h [cm]', fontsize=12)
axes[1].set_ylabel('Depth [cm]', fontsize=12)
axes[1].set_title('Pressure Head Profiles', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([-200, 0])

plt.tight_layout()
plt.show()

## 6. Time Series at Specific Depths

Evolution of state variables at selected depths - useful for monitoring specific soil layers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Select depths to monitor
depths_to_plot = [0, -25, -50, -75, -100]  # cm
colors = plt.cm.plasma(np.linspace(0, 1, len(depths_to_plot)))

if HAS_XARRAY and ds is not None:
    # Use xarray's powerful selection
    for depth, color in zip(depths_to_plot, colors):
        # Select nearest depth using xarray
        ds.theta.sel(depth=depth, method='nearest').plot(
            ax=axes[0],
            color=color,
            linewidth=2,
            label=f'z = {depth} cm'
        )
        ds.h.sel(depth=depth, method='nearest').plot(
            ax=axes[1],
            color=color,
            linewidth=2,
            label=f'z = {depth} cm'
        )
else:
    # Fallback to numpy indexing
    for depth, color in zip(depths_to_plot, colors):
        idx = np.argmin(np.abs(model.depths - depth))
        axes[0].plot(
            results['times'],
            results['theta'][:, idx],
            color=color,
            linewidth=2,
            label=f'z = {depth} cm'
        )
        axes[1].plot(
            results['times'],
            results['h'][:, idx],
            color=color,
            linewidth=2,
            label=f'z = {depth} cm'
        )

axes[0].set_xlabel('Time [days]', fontsize=12)
axes[0].set_ylabel('Water content θ [-]', fontsize=12)
axes[0].set_title('Water Content Time Series', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Time [days]', fontsize=12)
axes[1].set_ylabel('Pressure head h [cm]', fontsize=12)
axes[1].set_title('Pressure Head Time Series', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Surface (z=0) responds immediately to infiltration")
print("- Deeper layers show delayed response (wetting front arrival)")
print("- Bottom (z=-100) shows minimal change (wetting front hasn't reached yet)")

## 7. Mass Balance Analysis

Verify conservation of mass throughout the simulation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mb = results['mass_balance']
times = results['times']

# Cumulative fluxes
axes[0].plot(times, mb['flux_top'], 'b-', linewidth=2, label='Flux in (top)')
axes[0].plot(times, -mb['flux_bottom'], 'r-', linewidth=2, label='Flux out (bottom)')
axes[0].plot(times, mb['storage'] - mb['storage'][0], 'g-', linewidth=2, label='Storage change')
axes[0].set_xlabel('Time [days]', fontsize=12)
axes[0].set_ylabel('Cumulative water [cm]', fontsize=12)
axes[0].set_title('Mass Balance Components', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Mass balance error
rel_error = np.abs(mb['error']) / np.abs(mb['flux_top']) * 100
axes[1].plot(times, rel_error, 'k-', linewidth=2)
axes[1].axhline(1.0, color='g', linestyle='--', alpha=0.5, label='1% (excellent)')
axes[1].axhline(5.0, color='orange', linestyle='--', alpha=0.5, label='5% (good)')
axes[1].set_xlabel('Time [days]', fontsize=12)
axes[1].set_ylabel('Relative error [%]', fontsize=12)
axes[1].set_title('Mass Balance Error', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, max(10, rel_error.max() * 1.1)])

plt.tight_layout()
plt.show()

print(f"\nMass Balance Summary:")
print(f"  Final relative error: {rel_error[-1]:.2f}%")
if rel_error[-1] < 1.0:
    print(f"  ✓ Excellent mass balance (< 1%)")
elif rel_error[-1] < 5.0:
    print(f"  ✓ Good mass balance (< 5%)")
else:
    print(f"  ⚠ Consider refining discretization")

## 8. Advanced: Using xarray for Data Analysis

Demonstrate the power of xarray for data manipulation and analysis.

In [ ]:
if HAS_XARRAY and ds is not None:
    print("xarray Advanced Features:\n")
    
    # 1. Easy selection and slicing
    print("1. Select water content at 50 cm depth:")
    theta_50cm = ds.theta.sel(depth=-50, method='nearest')
    print(f"   Min: {theta_50cm.min().values:.3f}")
    print(f"   Max: {theta_50cm.max().values:.3f}")
    print(f"   Mean: {theta_50cm.mean().values:.3f}\n")
    
    # 2. Time-averaged profile
    print("2. Time-averaged water content profile:")
    theta_mean = ds.theta.mean(dim='time')
    print(f"   Shape: {theta_mean.shape}")
    print(f"   Overall mean: {theta_mean.mean().values:.3f}\n")
    
    # 3. Spatial-averaged time series
    print("3. Depth-averaged water content:")
    theta_profile_avg = ds.theta.mean(dim='depth')
    print(f"   Initial: {theta_profile_avg.values[0]:.3f}")
    print(f"   Final: {theta_profile_avg.values[-1]:.3f}")
    print(f"   Change: {theta_profile_avg.values[-1] - theta_profile_avg.values[0]:.3f}\n")
    
    # 4. Save to NetCDF
    output_file = Path.cwd() / 'infiltration_results.nc'
    ds.to_netcdf(output_file)
    print(f"4. Saved to NetCDF: {output_file}")
    print(f"   File size: {output_file.stat().st_size / 1024:.1f} KB")
    print(f"\n   To reload: ds = xr.open_dataset('{output_file.name}')")
    
else:
    print("Install xarray to enable these features:")
    print("  pip install xarray netcdf4")

## Summary

This notebook demonstrated:

1. **Modern data format**: xarray provides labeled arrays with metadata
2. **Spatiotemporal visualization**: Heatmaps reveal dynamics over time and depth
3. **Easy data manipulation**: Select by coordinate values, not array indices
4. **Flexible plotting**: Built-in plotting with proper labels
5. **Data persistence**: Save/load results in standard NetCDF format

### Key Advantages of xarray:
- Select by coordinate value: `ds.theta.sel(depth=-50)` vs. `results['theta'][:, 50]`
- Automatic labeling in plots
- Dimension-aware operations: `ds.theta.mean(dim='time')`
- Standard file format (NetCDF) for sharing and archiving
- Integration with pandas, dask, and other scientific Python tools